# Quick demo (≈1 h): where does Qwen2.5-VL lose Kannada spatial questions?

**Kaggle settings:** GPU **T4 x2** · Internet **on** · Inputs: `awsaf49/coco-2017-dataset` and your `xling-spatial-code` dataset.
Then *Run All*. Runtime ≈ 15–20 min (model download ~3 min, COCO boxes ~2 min, scoring ~5–10 min).

What it does:
1. Takes true English VSR statements about a COCO image whose objects and relation are in a small hand-written Hindi/Kannada
   table, and builds the Review-2 **minimal-pair triplet** on the same image, in all three languages:
   - original (true): *The cat is under the table* · ಬೆಕ್ಕು ಮೇಜಿನ ಕೆಳಗೆ ಇದೆ
   - relation flip (false): *The cat is above the table* · ಬೆಕ್ಕು ಮೇಜಿನ ಮೇಲೆ ಇದೆ (the postposition changes)
   - object swap (false): *The table is under the cat* · ಮೇಜು ಬೆಕ್ಕಿನ ಕೆಳಗೆ ಇದೆ (the genitive suffix moves to the other noun)

   A triplet counts as correct only if all three answers are right (pair accuracy).
2. Scores Qwen2.5-VL-3B on the same image + statement in EN / HI / KN (Yes/No logit margin).
3. Logs, per layer, where the answer position looks in the image, and how much of that lands on the two objects.
4. Applies the AdaptVis attention scaling (0.5 / 1.2 / 2.0) on all three languages.

**Caveat for the slides:** these Hindi/Kannada sentences are *template translations, not native-validated*. Present the numbers as a
preliminary pipeline run, not as benchmark results.

In [ ]:
# Shared setup for notebooks 02-04 (pinned transformers; do NOT run in the same session as notebook 01)
!pip install -q "transformers==4.57.1" "accelerate>=1.0" "huggingface_hub>=0.34"
import glob, os, sys
# The xling_spatial code: either a Kaggle dataset you uploaded (Add Input) or a git clone of your team repo.
REPO_URL = ""   # e.g. "https://github.com/<user>/<repo>.git" (leave empty if you attached the code as a Kaggle dataset)
def _find():
    return glob.glob("/kaggle/input/**/xling_spatial/__init__.py", recursive=True) + glob.glob("/kaggle/working/**/xling_spatial/__init__.py", recursive=True)
hits = _find()
if not hits and REPO_URL:
    !git clone -q {REPO_URL} /kaggle/working/repo
    hits = _find()
assert hits, "Set REPO_URL to your GitHub repo, or attach the 'xling-spatial-code' dataset (see README step 0)"
CODE_DIR = os.path.dirname(os.path.dirname(hits[0])); sys.path.insert(0, CODE_DIR); print("code:", CODE_DIR)

import torch, transformers, numpy as np, pandas as pd, matplotlib.pyplot as plt
from xling_spatial import data, runner, analysis, attention
from xling_spatial.models import VLM
from xling_spatial.tokenizer_stats import tokenizer_table
print(torch.__version__, transformers.__version__, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

COCO_ROOT = data.find_coco_root(); print("COCO:", COCO_ROOT)
WORK = "/kaggle/working"
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")   # needed only for gated models (Gemma 3)
except Exception:
    pass

def free(vlm):
    attention.uninstall()
    del vlm.model
    import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
from xling_spatial import templates
MINIMAL_PAIRS = True   # True: original / relation_flip / object_swap triplets (Review-2 design). False: plain VSR statements
N_PAIRS = 40           # 40 triplets = 120 statements x 3 languages x 4 weights ~ 1,450 forward passes (~10 min)
N_ITEMS = 90           # used only when MINIMAL_PAIRS = False
OUT = f"{WORK}/demo"; os.makedirs(OUT, exist_ok=True)

vsr = pd.concat([data.load_vsr("test", COCO_ROOT), data.load_vsr("val", COCO_ROOT)], ignore_index=True)
covered = templates.add_template_translations(vsr)
demo = templates.build_minimal_pairs(covered, N_PAIRS) if MINIMAL_PAIRS else templates.balanced_sample(covered, N_ITEMS)
print(f"VSR items: {len(vsr)} | covered by templates: {len(covered)} | demo set: {len(demo)} statements")
print(demo.relation_group.value_counts().to_dict(), "| true share:", demo.label.mean().round(2))
demo.to_csv(f"{OUT}/demo_items.csv", index=False)
demo[["pair_id", "variant", "label", "text_en", "text_hi", "text_kn"]].head(9)

In [ ]:
boxes = data.load_coco_boxes(COCO_ROOT, demo.image.unique().tolist(), f"{OUT}/boxes.json")
print("items with both object boxes:", np.mean([r.subj in boxes[r.image] and r.obj in boxes[r.image] for r in demo.itertuples()]).round(2))

In [ ]:
MODEL = "Qwen/Qwen2.5-VL-3B-Instruct"
qwen = VLM(MODEL)
tt = tokenizer_table({"Qwen2.5-VL": qwen.tokenizer}, demo, ["en", "hi", "kn"])
tt.to_csv(f"{OUT}/tokenizer.csv", index=False); tt.round(2)

In [ ]:
WEIGHTS = (1.0, 0.5, 1.2, 2.0)     # 1.0 = no change; others = AdaptVis image-attention scaling
res = runner.run(qwen, demo, ["en", "hi", "kn"], COCO_ROOT, f"{OUT}/run", weights=WEIGHTS, boxes=boxes, save_maps=True)
att = pd.read_csv(f"{OUT}/run/attention.csv", dtype={"item_id": str})
print("NaN margins:", res.margin.isna().sum(), "(if > 0: rerun with VLM(MODEL, dtype='float32'))")

## 1. How much accuracy drops

In [ ]:
s = analysis.summarize(res)
base = s[np.isclose(s.weight, 1.0)][["lang", "n", "acc", "pair_acc", "yes_rate", "mean_conf", "yesno_mass"]].set_index("lang")
base["drop_vs_en"] = base.acc - base.loc["en", "acc"]
av = {lang: runner.adaptvis_predictions(r, 0.5, 1.2, 0.64).correct.mean() for lang, r in res.groupby("lang")}
base["adaptvis (0.5/1.2/0.64)"] = pd.Series(av)
for w in (0.5, 1.2, 2.0):
    base[f"scale w={w}"] = s[np.isclose(s.weight, w)].set_index("lang").acc
base = base.loc[["en", "hi", "kn"]]
base.to_csv(f"{OUT}/accuracy.csv"); display(base.round(3))
by_variant = (res[np.isclose(res.weight, 1.0)].pivot_table(index="variant", columns="lang", values="correct")
              .reindex(columns=["en", "hi", "kn"]))
by_variant.to_csv(f"{OUT}/accuracy_by_variant.csv")
print("Accuracy by statement type (object_swap is where the Kannada genitive suffix moves):"); display(by_variant.round(3))

In [ ]:
# Slide-ready charts
langs = ["en", "hi", "kn"]; names = ["English", "Hindi", "Kannada"]; x = np.arange(3); wbar = 0.38
cols = [analysis.LANG_COLORS[l] for l in langs]
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.bar(x - wbar/2, base.loc[langs, "acc"], wbar, label="accuracy (each statement)", color=cols)
ax.bar(x + wbar/2, base.loc[langs, "pair_acc"], wbar, label="pair accuracy (whole triplet right)", color="none",
       edgecolor=cols, hatch="//", lw=1.5)
for i, l in enumerate(langs):
    ax.text(i - wbar/2, base.loc[l, "acc"] + 0.01, f"{base.loc[l, 'acc']:.2f}", ha="center", fontsize=9)
    ax.text(i + wbar/2, base.loc[l, "pair_acc"] + 0.01, f"{base.loc[l, 'pair_acc']:.2f}", ha="center", fontsize=9)
always_no = 1 - demo.label.mean()
ax.axhline(always_no, ls="--", color="grey", lw=1); ax.text(2.6, always_no + 0.01, "always 'no'", fontsize=8, color="grey", ha="right")
ax.set_xticks(x, names); ax.set_ylabel("score"); ax.set_ylim(0, 1.0)
ax.set_title(f"Qwen2.5-VL-3B · same image, same statement · {demo.pair_id.nunique()} triplets"); ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_accuracy.png", dpi=200); plt.show()

if "object_swap" in by_variant.index:
    fig, ax = plt.subplots(figsize=(6, 3.4)); order = ["original", "relation_flip", "object_swap"]; wv = 0.26
    for j, l in enumerate(langs):
        ax.bar(np.arange(3) + (j - 1) * wv, by_variant.loc[order, l], wv, label=names[j], color=cols[j])
    ax.set_xticks(np.arange(3), ["original (true)", "relation flip (false)", "object swap (false)"])
    ax.set_ylim(0, 1); ax.set_ylabel("accuracy"); ax.legend(frameon=False, fontsize=8)
    ax.set_title("Which statement type breaks in which language")
    fig.tight_layout(); fig.savefig(f"{OUT}/fig1b_accuracy_by_variant.png", dpi=200); plt.show()

## 2. Where the answer position looks, layer by layer (one line per language)

In [ ]:
fig = analysis.plot_layers(att, title="Qwen2.5-VL-3B · last-token attention over the image (all items)")
fig.savefig(f"{OUT}/fig2_layers_all.png", dpi=200); plt.show()

In [ ]:
findings = []
for lang in ("kn", "hi"):
    flip, both = analysis.flip_sets(res, lang)
    msg = analysis.diagnose(res, att, lang)
    findings.append(f"[{lang}] {len(flip)} items right in English but wrong in {lang}. {msg}")
    if len(flip) >= 3:
        fig = analysis.plot_layers(att[att.lang.isin(["en", lang])], items=flip,
                                   title=f"Only items right in English, wrong in {lang} (n={len(flip)})")
        fig.savefig(f"{OUT}/fig3_layers_flip_{lang}.png", dpi=200); plt.show()
    analysis.paired_layer_diff(att, "box_frac", lang, items=flip or None).to_csv(f"{OUT}/paired_box_frac_{lang}.csv", index=False)
open(f"{OUT}/findings.txt", "w").write("\n".join(findings)); print("\n\n".join(findings))

## 3. Heatmaps: the same image in English, Hindi and Kannada
Attention is logged identically for all three languages. Titles show the English statement because matplotlib has no Kannada font;
the exact Hindi/Kannada text the model saw is printed under each figure.

In [ ]:
flip, _ = analysis.flip_sets(res, "kn")
pick = sorted(flip)[:3] if flip else demo.item_id.head(2).tolist()
L = int(att.layer.max()) + 1; show = [L // 4, L // 2, 3 * L // 4]
for k, item in enumerate(pick):
    row = demo.set_index("item_id").loc[item]
    img = data.load_image(COCO_ROOT, row.image)
    for lang in ("en", "hi", "kn"):
        r = res[(res.item_id == item) & (res.lang == lang) & np.isclose(res.weight, 1.0)].iloc[0]
        fig = analysis.overlay(f"{OUT}/run/maps/{item}__{lang}.npz", img, layers=show,
                               title=f"[{lang.upper()}] \"{row.text_en}\"  ·  truth={'yes' if row.label else 'no'}, "
                                     f"model={'yes' if r.pred else 'no'}  (cyan=subject, green=object)")
        fig.savefig(f"{OUT}/fig4_heatmap_{k}_{lang}.png", dpi=170); plt.show()
        print(f"   {lang} text shown to the model: {row['text_' + lang]}")   # matplotlib has no Kannada font

In [ ]:
fig = analysis.confidence_by_lang(res); fig.savefig(f"{OUT}/fig5_confidence.png", dpi=200); plt.show()
fig = analysis.accuracy_vs_weight(res); fig.savefig(f"{OUT}/fig6_acc_vs_weight.png", dpi=200); plt.show()

In [ ]:
import shutil
shutil.make_archive(f"{WORK}/demo_outputs", "zip", OUT, ".")   # download this from the Output panel
print(sorted(f for f in os.listdir(OUT) if f.endswith((".png", ".csv", ".txt"))))

## What to put on the slides
1. **Pipeline** (one diagram): VSR image + statement → EN / HI / KN → Qwen2.5-VL (frozen) → Yes/No margin
   + per-layer attention logging + AdaptVis scaling hook. Say it runs on the other models through the same patch.
2. **fig1_accuracy + fig1b:** the cross-lingual drop on identical image–statement triplets (accuracy and pair accuracy),
   and which statement type fails: if Kannada fails mainly on object swaps, the model is not tracking which noun carries the suffix.
3. **fig2 / fig3:** where in the network Kannada attention to the two objects diverges from English (or does not).
4. **One heatmap pair (fig4):** the same image, English vs Kannada, at a middle layer.
5. **Tokenizer table:** tokens per word and parity vs English, so morphology isn't blamed for a tokenization effect.
6. **Caveats and next steps:** template translations (native-validated pilot in progress), one model, n≈90;
   next come the native pilot, three models, per-language AdaptVis tuning, then activation patching.